In [ ]:
# Cell 1: Setup parameters and Ops Audit Table
dbutils.widgets.text("catalog", "meridian_tnn")
catalog = dbutils.widgets.get("catalog")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.ops")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.dq_audit_log (
    check_name STRING,
    table_tested STRING,
    status STRING,
    failure_count BIGINT,
    executed_at TIMESTAMP
)
""")

failures = []

def run_dq_check(check_name, table_name, failure_query):
    bad_rows_df = spark.sql(failure_query)
    bad_count = bad_rows_df.count()
    status = "PASSED" if bad_count == 0 else "FAILED"
    
    spark.sql(f"""
    INSERT INTO {catalog}.ops.dq_audit_log VALUES 
    ('{check_name}', '{table_name}', '{status}', {bad_count}, current_timestamp())
    """)
    
    if status == "FAILED":
        failures.append(f"{check_name} on {table_name} detected {bad_count} bad rows")

# Check 1: Empty Gold Fact Verification
run_dq_check(
    check_name="empty_gold_fact_check",
    table_name="gold.fact_store_returns",
    failure_query=f"SELECT 1 WHERE (SELECT COUNT(*) FROM {catalog}.gold.fact_store_returns) = 0"
)

# Check 2: Primary Key / Ticket Uniqueness
run_dq_check(
    check_name="duplicate_ticket_item_check",
    table_name="gold.fact_store_returns",
    failure_query=f"""
        SELECT ticket_number, item_sk 
        FROM {catalog}.gold.fact_store_returns 
        GROUP BY ticket_number, item_sk 
        HAVING COUNT(*) > 1
    """
)

# Check 3: Orphan Fact Records (Referential Integrity to dim_store)
run_dq_check(
    check_name="orphan_fact_store_check",
    table_name="gold.fact_store_returns",
    failure_query=f"""
        SELECT f.store_sk 
        FROM {catalog}.gold.fact_store_returns f
        LEFT JOIN {catalog}.gold.dim_store s ON f.store_sk = s.s_store_sk
        WHERE s.s_store_sk IS NOT NULL
        AND s.s_store_sk IS NULL
    """
)

# Raise exception to fail the task if any check breached
if failures:
    raise Exception(f"DATA QUALITY GATE FAILURE: {'; '.join(failures)}")

print("All Data Quality Gate assertions passed successfully.")